# Libraries

In [ ]:
!pip install -q -U llmcompressor datasets accelerate transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.1 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.0 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)

from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

from huggingface_hub import (
    login,
    create_repo,
    upload_folder,
)

from kaggle_secrets import UserSecretsClient
from datasets import load_dataset

2026-05-03 11:30:44.137553: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777807844.537893      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777807844.657901      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777807845.666637      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777807845.666679      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777807845.666682      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [3]:
BASE_ID = "microsoft/Phi-4-mini-instruct"
MODEL_ID = "EdgeCompress01/Phi-4-mini-instruct-WANDA"
OUTPUT_DIR = "/kaggle/working/Phi-4-mini-instruct-WANDA-AWQ-4bit"
REPO_ID = f"EdgeCompress01/Phi-4-mini-instruct-WANDA-AWQ-4bit"
SEED = 42

# Functions

In [4]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

def format_chat(batch):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        for messages in batch["messages"]
    ]
    return {"text": texts}

# Set Seed

In [5]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface Login

In [6]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in")

Logged in


# Load Model & Tokenizer

In [ ]:
#Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    subfolder="Models/70",
    device_map="auto",
    dtype=torch.bfloat16
)

#Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_ID)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Models/25/model-00001-of-00002.safetenso(…):   0%|          | 0.00/4.90G [00:00<?, ?B/s]

Models/25/model-00002-of-00002.safetenso(…):   0%|          | 0.00/2.77G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

# AWQ Quantization

**Configurations**

In [8]:
recipe = [
    AWQModifier(
        ignore=["lm_head", "embed_tokens"], 
        scheme="W4A16",
        targets=["Linear"]
    )
]

**Calibration Dataset**

In [9]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").shuffle(seed=SEED).select(range(64))

formatted_dataset = dataset.map(
    format_chat,
    batched=True,
    remove_columns=dataset.column_names  # keep only "text"
)

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-f7dfac4afe5(…):   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

**Apply quantization**

In [ ]:
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=formatted_dataset,
    text_column="text",
    recipe=recipe,
    max_seq_length=512
)

2026-05-03T11:33:03.022072+0000 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


Tokenizing:   0%|          | 0/64 [00:00<?, ? examples/s]

2026-05-03T11:33:03.480254+0000 | _make_sampler | WARNING - Requested 512 samples but the provided dataset only has 64 samples.
2026-05-03T11:33:03.481547+0000 | reset | INFO - Compression lifecycle reset
2026-05-03T11:33:03.483976+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-05-03T11:33:03.557847+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-05-03T11:33:03.565542+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.qkv_proj for mapping AWQMapping(smooth_layer='re:.*qkv_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-05-03T11:33:03.566713+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.1.self_attn.qkv_proj for mapping AWQMapping(smooth_layer='re:.*qkv_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-05-03T11:33:03.567835+0000 | _

W0503 11:33:04.108000 57 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Smoothing:  33%|███▎      | 1/3 [00:08<00:16,  8.33s/it]                                                             
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:00<?, ?it/s]
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:01<?, ?it/s, best_error=2.232e-04]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:01<00:23,  1.26s/it, best_error=2.232e-04]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:02<00:23,  1.26s/it, best_error=2.222e-04]
Grid search for model.layers.0.post_attention_layernorm:  10%|█         | 2/20 [00:02<00:22,  1.26s/it, best_error=2.222e-04]
Grid s

**Saving**

In [ ]:
model.config._name_or_path = ""

model.save_pretrained(OUTPUT_DIR,skip_sparsity_compression_stats=False)#VERY IMPORTANT
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully")

# PUSH TO HUGGING FACE

In [ ]:
create_repo(REPO_ID, repo_type="model", exist_ok=True)

upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    path_in_repo="Sparsity/70"
)